# Learning mFISH — ophys metadata + QC tables via docDB

Builds the session-level, per-plane, and QC-flag tables for the Learning mFISH /
CTL cohort from the AIND metadata docDB, following the conventions in
`bci_metadata.ipynb` / `V1DD_metadata.ipynb`.

Everything here comes from docDB alone — no NWB-Zarr reads and no S3 object
listings.

**Coverage, measured against the 2026-08-19 batch of 158 processed assets:**

| | docDB | note |
|---|---|---|
| assets indexed | 130 / 158 | 28 assets from the same batch were not indexed |
| assets with a `quality_control` block | 116 / 130 | 14 indexed docs carry no QC at all |
| session / plane / QC field values | identical to the asset JSONs | 0 disagreements where both exist |

docDB is a faithful mirror of the asset metadata, so the risk is not wrong
values — it is **silently missing rows**. Section 4 checks coverage against an
authoritative asset list; run it before trusting any output.

`metadata_status` is `"Invalid"` on all 130 indexed docs. That reflects
schema-validation state, not data quality, so it cannot be used as a filter.

In [ ]:
import pandas as pd
from datetime import datetime
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)

In [ ]:
MICE = ["782149", "790322", "788406", "800792", "800995", "804363"]
PROCESSING_DATE = "2026-08-19"   # set to the batch you want to audit

match = {"$match": {"name": {"$regex": "^multiplane-ophys_(%s)_.*_processed_%s"
                                       % ("|".join(MICE), PROCESSING_DATE)}}}

## 1. Session-level table

In [ ]:
aggregate = [
  match,
  {
    "$project": {
      "name": 1,
      "location": 1,
      "project_name": "$data_description.project_name",
      "data_level": "$data_description.data_level",
      "modality": "$data_description.modality.abbreviation",
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.genotype",
      "sex": "$subject.sex",
      "date_of_birth": "$subject.date_of_birth",
      "species": "$subject.species.name",
      "session_type": "$session.session_type",
      "session_start_time": "$session.session_start_time",
      "session_end_time": "$session.session_end_time",
      "rig_id": "$session.rig_id",
      "mouse_platform": "$session.mouse_platform_name",
      "stimulus_epochs": "$session.stimulus_epochs.stimulus_name",
      "targeted_structure": {"$arrayElemAt": ["$session.data_streams.ophys_fovs.targeted_structure", 0]},
      "imaging_depths": {"$arrayElemAt": ["$session.data_streams.ophys_fovs.imaging_depth", 0]},
      "coupled_fov_index": {"$arrayElemAt": ["$session.data_streams.ophys_fovs.coupled_fov_index", 0]},
      "frame_rate": {"$arrayElemAt": ["$session.data_streams.ophys_fovs.frame_rate", 0]},
      "pipeline_version": "$processing.processing_pipeline.pipeline_version",
      "n_data_processes": {"$size": {"$ifNull": ["$processing.processing_pipeline.data_processes", []]}},
    }
  },
]
records = docdb_api_client.aggregate_docdb_records(pipeline=aggregate)
print(len(records), "assets indexed")

In [ ]:
df = pd.DataFrame(records).drop_duplicates(subset="name")

df['session_date'] = df.apply(lambda x: datetime.fromisoformat(x['session_start_time']).date(), axis=1)
df['session_end_time'] = df.apply(
    lambda x: datetime.fromisoformat(x['session_end_time']).time() if pd.notna(x['session_end_time']) else None, axis=1)
df['session_start_time'] = df.apply(lambda x: datetime.fromisoformat(x['session_start_time']).time(), axis=1)
df['date_of_birth'] = df.apply(lambda x: datetime.strptime(x['date_of_birth'], '%Y-%m-%d').date(), axis=1)
df['age'] = df.apply(lambda x: (x['session_date'] - x['date_of_birth']).days, axis=1)

df['n_planes'] = df.imaging_depths.apply(len)
df['imaging_depths_um'] = df.imaging_depths.apply(lambda d: "|".join(str(x) for x in sorted(d)))
df['depth_min_um'] = df.imaging_depths.apply(min)
df['depth_max_um'] = df.imaging_depths.apply(max)
df['n_coupled_pairs'] = df.coupled_fov_index.apply(lambda c: len(set(c)))
df['targeted_structure'] = df.targeted_structure.apply(lambda t: "|".join(sorted(set(t))))
df['frame_rate_hz'] = df.frame_rate.apply(lambda f: "|".join(sorted(set(map(str, f)))))
df['modality'] = df.modality.apply(lambda m: "|".join(m) if isinstance(m, list) else m)
df['stimulus_epochs'] = df.stimulus_epochs.apply(
    lambda s: "|".join(dict.fromkeys(map(str, s)))[:200] if isinstance(s, list) else s)
df['processing_date'] = df.name.str.extract(r'_processed_(\d{4}-\d\d-\d\d)_')[0]

df = df.sort_values(['subject_id', 'session_date', 'session_start_time']).reset_index(drop=True)
df['session_number'] = df.groupby('subject_id').cumcount() + 1

order = ['project_name', 'session_type', '_id', 'name', 'subject_id', 'genotype',
         'date_of_birth', 'age', 'sex', 'species', 'modality', 'session_number',
         'session_date', 'session_start_time', 'session_end_time', 'rig_id', 'mouse_platform',
         'targeted_structure', 'n_planes', 'imaging_depths_um', 'depth_min_um', 'depth_max_um',
         'n_coupled_pairs', 'frame_rate_hz', 'stimulus_epochs',
         'data_level', 'pipeline_version', 'n_data_processes', 'processing_date', 'location']
session_df = df[order]
session_df.head()

## 2. Per-plane table

`$unwind` twice: once over `data_streams`, once over `ophys_fovs`.

In [ ]:
aggregate_planes = [
  match,
  {"$unwind": "$session.data_streams"},
  {"$unwind": "$session.data_streams.ophys_fovs"},
  {
    "$project": {
      "name": 1,
      "project_name": "$data_description.project_name",
      "subject_id": "$data_description.subject_id",
      "genotype": "$subject.genotype",
      "sex": "$subject.sex",
      "date_of_birth": "$subject.date_of_birth",
      "session_type": "$session.session_type",
      "session_start_time": "$session.session_start_time",
      "rig_id": "$session.rig_id",
      "fov_index": "$session.data_streams.ophys_fovs.index",
      "targeted_structure": "$session.data_streams.ophys_fovs.targeted_structure",
      "imaging_depth_um": "$session.data_streams.ophys_fovs.imaging_depth",
      "scanfield_z_um": "$session.data_streams.ophys_fovs.scanfield_z",
      "coupled_fov_index": "$session.data_streams.ophys_fovs.coupled_fov_index",
      "frame_rate_hz": "$session.data_streams.ophys_fovs.frame_rate",
    }
  },
]
plane_records = docdb_api_client.aggregate_docdb_records(pipeline=aggregate_planes)
plane_df = pd.DataFrame(plane_records)
plane_df['session_id'] = plane_df.name.str.split('_processed_').str[0]
plane_df['plane'] = 'VISp_' + plane_df.fov_index.astype(str)
print(len(plane_df), "plane rows")
plane_df.head()

## 3. QC tables

`quality_control.evaluations[].metrics[]` is doubly nested, so unwind twice.
Each metric's scope is encoded in its *name* (`VISp_3 FOV Quality`,
`Coupled FOV 2`, `Vasculature_image`) — parse it out to get per-plane rows.

In [ ]:
aggregate_qc = [
  match,
  {"$unwind": "$quality_control.evaluations"},
  {"$unwind": "$quality_control.evaluations.metrics"},
  {
    "$project": {
      "name": 1,
      "stage": "$quality_control.evaluations.stage",
      "modality": "$quality_control.evaluations.modality.abbreviation",
      "evaluation": "$quality_control.evaluations.name",
      "evaluation_description": "$quality_control.evaluations.description",
      "evaluation_tags": "$quality_control.evaluations.tags",
      "allow_failed_metrics": "$quality_control.evaluations.allow_failed_metrics",
      "eval_latest_status": "$quality_control.evaluations.latest_status",
      "metric": "$quality_control.evaluations.metrics.name",
      "metric_description": "$quality_control.evaluations.metrics.description",
      "reference": "$quality_control.evaluations.metrics.reference",
      "value_type": "$quality_control.evaluations.metrics.value.type",
      "value_options": "$quality_control.evaluations.metrics.value.options",
      "option_statuses": "$quality_control.evaluations.metrics.value.status",
      "status_history": "$quality_control.evaluations.metrics.status_history",
    }
  },
]
qc_records = docdb_api_client.aggregate_docdb_records(pipeline=aggregate_qc)
qc_df = pd.DataFrame(qc_records)
qc_df['metric_status'] = qc_df.status_history.apply(lambda h: h[-1]['status'] if h else None)
qc_df['evaluator'] = qc_df.status_history.apply(lambda h: h[-1].get('evaluator') if h else None)
qc_df['session_id'] = qc_df.name.str.split('_processed_').str[0]
print(len(qc_df), "metric rows from", qc_df.name.nunique(), "assets")

In [ ]:
import re

def metric_scope(metric_name):
    """Resolve a metric to the entity it scores: an imaging plane, a coupled
    plane pair, or the whole session."""
    planes = re.findall(r'VISp_(\d)', metric_name)
    if planes:
        return f'VISp_{planes[0]}'
    pair = re.match(r'Coupled FOV (\d)', metric_name)
    if pair:
        return f'coupled_pair_{pair.group(1)}'
    return 'SESSION'

def canonical_metric(metric_name):
    """Strip the plane token so the same check across planes shares one name."""
    s = re.sub(r'VISp_\d', '<plane>', metric_name)
    s = re.sub(r'Coupled FOV \d', 'Coupled FOV <pair>', s)
    s = re.sub(r'<plane>\s*(.*?)\s*-\s*<plane>', r'<plane> \1', s).strip()
    s = re.sub(r'\s*-\s*<plane>$', '', s).strip()
    return re.sub(r'\s+', ' ', s)

qc_df['scope'] = qc_df.metric.apply(metric_scope)
qc_df['metric_canonical'] = qc_df.metric.apply(canonical_metric)
qc_df.groupby(['evaluation', 'metric_canonical']).scope.apply(lambda s: sorted(set(s))[:3])

### 3a. QC flag dictionary — one row per (evaluation, canonical metric)

In [ ]:
def first_n(s, n=2, sep=" || "):
    vals = [x for x in dict.fromkeys(s.dropna().astype(str)) if x and x != 'nan']
    return sep.join(vals[:n]) + (f" [+{len(vals)-n} more]" if len(vals) > n else "")

flag_dict = qc_df.groupby(['stage', 'evaluation', 'metric_canonical']).apply(
    lambda d: pd.Series({
        'modality': first_n(d.modality, 1),
        'scope': ('per imaging plane' if d.scope.str.startswith('VISp').all()
                  else 'per coupled plane pair' if d.scope.str.startswith('coupled').all()
                  else 'per session'),
        'evaluation_description': first_n(d.evaluation_description),
        'metric_description': first_n(d.metric_description),
        'value_type': first_n(d.value_type),
        'value_options': first_n(d.value_options.astype(str), 1),
        'option_statuses': first_n(d.option_statuses.astype(str), 1),
        'allow_failed_metrics': first_n(d.allow_failed_metrics.astype(str)),
        'evaluation_tags': first_n(d.evaluation_tags.astype(str)),
        'reference_example': first_n(d.reference, 1),
        'n_records': len(d),
        'n_assets': d.name.nunique(),
        'n_pass': int((d.metric_status == 'Pass').sum()),
        'n_fail': int((d.metric_status == 'Fail').sum()),
        'n_pending': int((d.metric_status == 'Pending').sum()),
    }), include_groups=False).reset_index()
flag_dict

### 3b. Per-plane QC status, joined onto the plane table

In [ ]:
plane_qc = qc_df[qc_df.scope.str.startswith('VISp')].copy()
plane_qc['plane'] = plane_qc.scope

status_wide = plane_qc.pivot_table(
    index=['session_id', 'plane'], columns='metric_canonical', values='metric_status',
    aggfunc=lambda s: 'Fail' if (s == 'Fail').any() else ('Pass' if (s == 'Pass').any() else 'Pending'))
status_wide.columns = ['qc_' + re.sub(r'[^A-Za-z0-9]+', '_', c.replace('<plane> ', '')).strip('_').lower()
                       for c in status_wide.columns]

qc_summary = plane_qc.groupby(['session_id', 'plane']).apply(lambda d: pd.Series({
    'qc_n_metrics': len(d),
    'qc_n_pass': int((d.metric_status == 'Pass').sum()),
    'qc_n_fail': int((d.metric_status == 'Fail').sum()),
    'qc_n_pending': int((d.metric_status == 'Pending').sum()),
    'qc_failed_metrics': "|".join(sorted(set(d.loc[d.metric_status == 'Fail', 'metric_canonical']))),
    'qc_evaluations_present': "|".join(sorted(set(d.evaluation))),
}), include_groups=False)
qc_summary['qc_any_fail'] = qc_summary.qc_n_fail > 0

# crosstalk is scored per coupled PAIR — map it onto each member plane
pair_qc = qc_df[qc_df.scope.str.startswith('coupled')].copy()
pair_qc['pair'] = pair_qc.scope.str.extract(r'(\d)$').astype(int)
pair_status = dict(zip(zip(pair_qc.session_id, pair_qc.pair), pair_qc.metric_status))

plane_full = (plane_df
              .merge(qc_summary.reset_index(), on=['session_id', 'plane'], how='left')
              .merge(status_wide.reset_index(), on=['session_id', 'plane'], how='left'))
plane_full['qc_crosstalk_pair_status'] = [
    pair_status.get((sid, ci)) for sid, ci in zip(plane_full.session_id, plane_full.coupled_fov_index)]
print(plane_full.shape, '|', int(plane_full.qc_any_fail.fillna(False).sum()), 'planes with >=1 failing metric')
plane_full.head()

## 4. Coverage check — always run this

docDB indexing lags asset creation, and some indexed docs carry no
`quality_control` block. Both failure modes are silent: the aggregation simply
returns fewer rows. Get the authoritative asset list from Code Ocean and diff
against it.

For the 2026-08-19 batch this surfaced 44 assets with metadata issues, confined
to three mice (788406, 790322, 800792) — see
`metadata_issue_assets_2026-08-19.csv`. The gaps nest: every unindexed asset also
lacked `original_metadata`, and so did every asset indexed without QC.

In [ ]:
# Authoritative asset list from Code Ocean (codeocean MCP connector or the
# codeocean Python client). Fill `expected_assets` with the asset names that
# SHOULD be present for this batch.
expected_assets = set()   # e.g. {a["name"] for a in codeocean_search_results}

indexed = set(df.name)
with_qc = set(qc_df.name)

print(f"indexed in docDB   : {len(indexed)}")
print(f"of those, with QC  : {len(with_qc)}")
print(f"indexed but no QC  : {len(indexed - with_qc)}")

if expected_assets:
    not_indexed = expected_assets - indexed
    print(f"expected           : {len(expected_assets)}")
    print(f"NOT indexed        : {len(not_indexed)}")

    issues = (
        [{"name": n, "issue_type": "not_indexed_in_docdb",
          "indexed_in_docdb": False, "has_quality_control_in_docdb": False} for n in sorted(not_indexed)]
        + [{"name": n, "issue_type": "indexed_no_quality_control",
            "indexed_in_docdb": True, "has_quality_control_in_docdb": False}
           for n in sorted(indexed - with_qc)]
    )
    issue_df = pd.DataFrame(issues)
    if len(issue_df):
        issue_df["subject_id"] = issue_df.name.str.extract(r"multiplane-ophys_(\d+)_")
        issue_df["session_id"] = issue_df.name.str.split("_processed_").str[0]
        issue_df = issue_df.merge(
            session_df[["name", "session_type", "session_date"]], on="name", how="left")
        issue_df.to_csv("metadata_issue_assets.csv", index=False)
    display(issue_df)
else:
    print("\nexpected_assets is empty — fill it in, or these tables may be silently incomplete.")

## 5. Caveat on `stimulus_epochs`

`session.stimulus_epochs` is empty in the source metadata for a subset of
sessions (10 of 158 in the 2026-08-19 batch), so the `stimulus_epochs` column is
blank for those. This is an acquisition-side gap present in the raw asset, not
something docDB or the processing pipeline dropped. `session.session_type`
remains populated and is the reliable field for what protocol ran.

`trials_total` / `trials_rewarded` are null in `stimulus_epochs` for every
session in this cohort, so trial counts are not available from metadata.

In [ ]:
session_df.to_csv('learning_mfish_session_metadata.csv', index=False)
plane_full.to_csv('learning_mfish_session_plane_metadata.csv', index=False)
flag_dict.to_csv('learning_mfish_qc_flag_dictionary.csv', index=False)
qc_df.drop(columns=['status_history']).to_csv('learning_mfish_qc_metrics_long.csv', index=False)